# 01 - Data Pipeline

Task 1 demonstrates a reproducible, leakage-safe pipeline for the UCI AI4I 2020 Predictive Maintenance dataset. It loads the raw CSV, inspects data quality, applies conservative structural cleaning, validates the result, and writes `data/processed/processed_data.csv`. No model-specific scaling, encoding, resampling, feature selection, or train/test preprocessing is performed here.


## 1. Project / Data Pipeline Overview

**Flow:** raw CSV -> reusable loader -> structural cleaning -> validation -> processed CSV.

The project brief specifies AI4I 2020. The source contains identifiers, L/M/H product type, five operating measurements, the `Machine failure` target, and five failure-mode labels. The failure-mode labels are retained for analysis but must not be used as predictors of `Machine failure` in later ML work because that would create target leakage.


## 2. Imports and project-relative paths


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.data.load_data import load_raw_data, save_processed_data
from src.data.clean_data import clean_data
from src.data.validate_data import run_validation

raw_path = ROOT / "data" / "raw" / "ai4i2020.csv"
processed_path = ROOT / "data" / "processed" / "processed_data.csv"
print("Project root:", ROOT)
print("Raw path:", raw_path)
print("Processed path:", processed_path)


Project root: C:\Users\dell\Downloads\industrial-predictive-maintenance-task1-completed\industrial-predictive-maintenance-main
Raw path: C:\Users\dell\Downloads\industrial-predictive-maintenance-task1-completed\industrial-predictive-maintenance-main\data\raw\ai4i2020.csv
Processed path: C:\Users\dell\Downloads\industrial-predictive-maintenance-task1-completed\industrial-predictive-maintenance-main\data\processed\processed_data.csv


## 3. Load Raw Data


In [2]:
df_raw = load_raw_data(raw_path)
print(f"Raw shape: {df_raw.shape}")
df_raw.head()


Raw shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## 4. Initial Dataset Inspection


In [3]:
df_raw.info()
display(df_raw.describe(include="all").T)


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
UDI,10000.0,NaN,NaN,NaN,5000.5,2886.89568,1.0,2500.75,5000.5,7500.25,10000.0
Product ID,10000,10000,M14860,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Type,10000,3,L,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Air temperature [K],10000.0,NaN,NaN,NaN,300.00493,2.000259,295.3,298.3,300.1,301.5,304.5
Process temperature [K],10000.0,NaN,NaN,NaN,310.00556,1.483734,305.7,308.8,310.1,311.1,313.8
Rotational speed [rpm],10000.0,NaN,NaN,NaN,1538.7761,179.284096,1168.0,1423.0,1503.0,1612.0,2886.0
Torque [Nm],10000.0,NaN,NaN,NaN,39.98691,9.968934,3.8,33.2,40.1,46.8,76.6
Tool wear [min],10000.0,NaN,NaN,NaN,107.951,63.654147,0.0,53.0,108.0,162.0,253.0
Machine failure,10000.0,NaN,NaN,NaN,0.0339,0.180981,0.0,0.0,0.0,0.0,1.0
TWF,10000.0,NaN,NaN,NaN,0.0046,0.067671,0.0,0.0,0.0,0.0,1.0


## 5. Data Quality Analysis


In [4]:
quality_summary = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (100 * df_raw.isna().mean()).round(3),
    "unique_values": df_raw.nunique(dropna=False),
})
display(quality_summary)

print("Exact duplicate rows:", int(df_raw.duplicated().sum()))
print("Unique Type values:", sorted(df_raw["Type"].dropna().astype(str).unique().tolist()))
print("Machine failure distribution:")
display(df_raw["Machine failure"].value_counts(dropna=False).sort_index().to_frame("rows"))


,dtype,missing_count,missing_pct,unique_values
UDI,int64,0,0.0,10000
Product ID,str,0,0.0,10000
Type,str,0,0.0,3
Air temperature [K],float64,0,0.0,93
Process temperature [K],float64,0,0.0,82
Rotational speed [rpm],int64,0,0.0,941
Torque [Nm],float64,0,0.0,577
Tool wear [min],int64,0,0.0,246
Machine failure,int64,0,0.0,2
TWF,int64,0,0.0,2


Exact duplicate rows: 0
Unique Type values: ['H', 'L', 'M']
Machine failure distribution:


,rows
Machine failure,
0,9661
1,339


### Source-label consistency note

The validation layer also reports, as a **warning rather than an error**, published records where `Machine failure` differs from the simple OR of `TWF/HDF/PWF/OSF/RNF`. Task 1 preserves source labels instead of rewriting target data.


## 6. Cleaning


In [5]:
df_clean = clean_data(df_raw)
print(f"Raw shape:   {df_raw.shape}")
print(f"Clean shape: {df_clean.shape}")
print(f"Rows removed (exact duplicates only): {len(df_raw) - len(df_clean)}")
df_clean.head()


Raw shape:   (10000, 14)
Clean shape: (10000, 14)
Rows removed (exact duplicates only): 0


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


Cleaning is deliberately conservative. It trims structural whitespace, drops accidental `Unnamed:*` export-index columns, normalizes `Product ID`/`Type` formatting, coerces documented numeric fields, and removes exact duplicate rows. It does not impute, scale, encode, clip, or use target values to remove records.


## 7. Validation


In [6]:
validation_report = run_validation(df_clean)
print("Passed:", validation_report["passed"])
print("Rows / columns:", validation_report["rows"], "/", validation_report["columns"])

if validation_report["errors"]:
    print("Errors:")
    for message in validation_report["errors"]:
        print(" -", message)

if validation_report["warnings"]:
    print("Warnings:")
    for message in validation_report["warnings"]:
        print(" -", message)

validation_report


Passed: True
Rows / columns: 10000 / 14
Warnings:
 - 27 published rows have Machine failure different from the OR of the five failure-mode flags. These labels are preserved as source data, not rewritten by the pipeline.


{'passed': True,
 'rows': 10000,
 'columns': 14,
 'errors': [],
 'warnings': ['27 published rows have Machine failure different from the OR of the five failure-mode flags. These labels are preserved as source data, not rewritten by the pipeline.']}

## 8. Save Processed Dataset


In [7]:
if not validation_report["passed"]:
    raise ValueError(f"Validation failed: {validation_report['errors']}")

saved_path = save_processed_data(df_clean, processed_path)
print("Saved:", saved_path)


Saved: C:\Users\dell\Downloads\industrial-predictive-maintenance-task1-completed\industrial-predictive-maintenance-main\data\processed\processed_data.csv


## 9. Final Dataset Summary


In [8]:
final_summary = pd.DataFrame({
    "rows": [len(df_clean)],
    "columns": [len(df_clean.columns)],
    "machine_failures": [int(df_clean["Machine failure"].sum())],
    "failure_rate_pct": [round(100 * df_clean["Machine failure"].mean(), 3)],
    "validation_passed": [validation_report["passed"]],
})
display(final_summary)
display(df_clean.dtypes.astype(str).to_frame("dtype"))

print("Handoff file for Person 2 / Person 3:", processed_path)
print("Leakage reminder: do not use TWF/HDF/PWF/OSF/RNF as predictors of Machine failure.")


,rows,columns,machine_failures,failure_rate_pct,validation_passed
0,10000,14,339,3.39,True


,dtype
UDI,int64
Product ID,string
Type,string
Air temperature [K],float64
Process temperature [K],float64
Rotational speed [rpm],int64
Torque [Nm],float64
Tool wear [min],int64
Machine failure,int64
TWF,int64


Handoff file for Person 2 / Person 3: C:\Users\dell\Downloads\industrial-predictive-maintenance-task1-completed\industrial-predictive-maintenance-main\data\processed\processed_data.csv
Leakage reminder: do not use TWF/HDF/PWF/OSF/RNF as predictors of Machine failure.
